# 10 — Advanced Topics

These topics come up in senior-level interviews and show deep Node.js expertise.

---

## Table of Contents
1. Worker Threads
2. Child Processes
3. Design Patterns
4. Microservices Architecture
5. Message Queues
6. WebSockets & Real-Time
7. GraphQL vs REST
8. TypeScript with Node.js
9. Node.js Best Practices Summary
10. Interview Questions

---
## 1. Worker Threads

Worker Threads allow you to run JavaScript in **parallel threads** — perfect for CPU-intensive tasks without blocking the event loop.

### When to use Worker Threads:
- Image/video processing
- Complex calculations
- Data parsing (large CSV/JSON)
- Encryption/compression

### When NOT to use:
- I/O operations (the event loop already handles these efficiently)
- Simple tasks (overhead of creating a thread isn't worth it)

In [ ]:
const { Worker, isMainThread, parentPort, workerData } = require('worker_threads');

if (isMainThread) {
    // Main thread — create a worker
    console.log('Main thread: starting heavy computation in worker...');

    // In real code, you'd pass a file path:
    // const worker = new Worker('./worker.js', { workerData: { n: 40 } });

    // Inline worker for demonstration
    const worker = new Worker(`
        const { parentPort, workerData } = require('worker_threads');
        function fibonacci(n) {
            if (n <= 1) return n;
            return fibonacci(n - 1) + fibonacci(n - 2);
        }
        const result = fibonacci(workerData.n);
        parentPort.postMessage(result);
    `, { eval: true, workerData: { n: 35 } });

    worker.on('message', (result) => {
        console.log(`Main thread: worker result = ${result}`);
    });

    worker.on('error', (err) => console.error('Worker error:', err));
    worker.on('exit', (code) => console.log(`Worker exited with code ${code}`));

    console.log('Main thread: still responsive while worker computes!');
}

In [ ]:
// Worker Thread Pool pattern (reuse workers)
const { Worker } = require('worker_threads');

class WorkerPool {
    constructor(workerScript, size) {
        this.workers = [];
        this.queue = [];
        for (let i = 0; i < size; i++) {
            this.workers.push({ busy: false, index: i });
        }
    }

    async execute(data) {
        return new Promise((resolve, reject) => {
            const available = this.workers.find(w => !w.busy);
            if (available) {
                this._runWorker(available, data, resolve, reject);
            } else {
                this.queue.push({ data, resolve, reject });
            }
        });
    }

    _runWorker(workerInfo, data, resolve, reject) {
        workerInfo.busy = true;
        console.log(`  Worker ${workerInfo.index} processing...`);
        setTimeout(() => { // Simulating worker
            workerInfo.busy = false;
            resolve(data * 2);
            if (this.queue.length > 0) {
                const next = this.queue.shift();
                this._runWorker(workerInfo, next.data, next.resolve, next.reject);
            }
        }, 100);
    }
}

const pool = new WorkerPool('worker.js', 2);
Promise.all([1, 2, 3, 4, 5].map(n => pool.execute(n)))
    .then(results => console.log('Pool results:', results));

### Worker Threads vs Child Processes vs Cluster:

| Feature | Worker Threads | Child Processes | Cluster |
|---------|--------------|----------------|--------|
| Memory | Shared (SharedArrayBuffer) | Separate | Separate |
| Communication | postMessage + SharedArrayBuffer | IPC (serialized) | IPC |
| Overhead | Low | High (new V8 instance) | High |
| Use case | CPU-intensive JS | Run external programs, isolation | Scale HTTP server |
| Module | `worker_threads` | `child_process` | `cluster` |

---
## 2. Child Processes

Run external commands or scripts in separate processes.

In [ ]:
const { exec, execSync, spawn, fork } = require('child_process');

// exec — runs command in shell, buffers output
exec('node --version', (err, stdout, stderr) => {
    if (err) return console.error(err);
    console.log('exec:', stdout.trim());
});

// execSync — synchronous version
const version = execSync('node --version').toString().trim();
console.log('execSync:', version);

// spawn — streams output (for long-running processes)
const ls = spawn('node', ['-e', 'console.log("from spawn")']);
ls.stdout.on('data', (data) => console.log('spawn:', data.toString().trim()));
ls.on('close', (code) => console.log('spawn exited with code', code));

// fork — special spawn for Node.js scripts (built-in IPC)
// const child = fork('./child-script.js');
// child.send({ type: 'task', data: 'work' });
// child.on('message', (msg) => console.log('From child:', msg));

### exec vs spawn vs fork:

| Method | Shell | Buffered | IPC | Best for |
|--------|-------|----------|-----|----------|
| `exec` | Yes | Yes (max buffer) | No | Short commands with small output |
| `spawn` | No | No (streams) | No | Long-running processes, large output |
| `fork` | No | No (streams) | Yes (built-in) | Node.js child scripts with messaging |

---
## 3. Design Patterns

### Patterns frequently asked in Node.js interviews:

In [ ]:
// 1. SINGLETON — module caching makes this natural in Node.js
class Database {
    constructor() {
        if (Database.instance) return Database.instance;
        this.connection = 'connected';
        Database.instance = this;
    }
}

const db1 = new Database();
const db2 = new Database();
console.log('Singleton - same instance:', db1 === db2); // true

// Even simpler with modules:
// db.js:
// module.exports = new Database();  ← cached by require()

In [ ]:
// 2. FACTORY — create objects without specifying exact class
class PostgresDB { query() { return 'PG result'; } }
class MongoDB { query() { return 'Mongo result'; } }
class SQLiteDB { query() { return 'SQLite result'; } }

function createDatabase(type) {
    switch (type) {
        case 'postgres': return new PostgresDB();
        case 'mongodb': return new MongoDB();
        case 'sqlite': return new SQLiteDB();
        default: throw new Error(`Unknown database type: ${type}`);
    }
}

const db = createDatabase('postgres');
console.log('Factory:', db.query());

In [ ]:
// 3. OBSERVER / PUB-SUB — built into Node.js via EventEmitter!
const EventEmitter = require('events');

class OrderService extends EventEmitter {
    placeOrder(order) {
        console.log('Order placed:', order.id);
        this.emit('orderPlaced', order);
    }
}

const orderService = new OrderService();

// Subscribers (decoupled from OrderService)
orderService.on('orderPlaced', (order) => console.log('  → Email sent for order', order.id));
orderService.on('orderPlaced', (order) => console.log('  → Inventory updated for order', order.id));
orderService.on('orderPlaced', (order) => console.log('  → Analytics tracked for order', order.id));

orderService.placeOrder({ id: 'ORD-001', total: 99.99 });

In [ ]:
// 4. MIDDLEWARE / CHAIN OF RESPONSIBILITY — Express uses this!
function createPipeline() {
    const handlers = [];
    return {
        use(fn) { handlers.push(fn); return this; },
        async execute(context) {
            for (const handler of handlers) {
                await handler(context);
                if (context.stopped) break;
            }
            return context;
        }
    };
}

const pipeline = createPipeline();
pipeline
    .use(async (ctx) => { ctx.steps = []; ctx.steps.push('validate'); })
    .use(async (ctx) => { ctx.steps.push('transform'); })
    .use(async (ctx) => { ctx.steps.push('save'); });

pipeline.execute({}).then(ctx => console.log('Pipeline:', ctx.steps));

In [ ]:
// 5. DEPENDENCY INJECTION — essential for testability

// WITHOUT DI — tightly coupled, hard to test
// class UserService {
//     constructor() {
//         this.db = require('./database'); // Hard dependency
//     }
// }

// WITH DI — loosely coupled, easy to test
class UserService {
    constructor(db, logger) {
        this.db = db;
        this.logger = logger;
    }

    async getUser(id) {
        this.logger.log(`Fetching user ${id}`);
        return this.db.findById(id);
    }
}

// Production
const realService = new UserService(
    { findById: (id) => ({ id, name: 'Alice' }) },
    { log: console.log }
);

// Test — inject mocks!
const mockDb = { findById: () => ({ id: 1, name: 'Mock' }) };
const mockLogger = { log: () => {} };
const testService = new UserService(mockDb, mockLogger);

realService.getUser(1).then(u => console.log('Real:', u));
testService.getUser(1).then(u => console.log('Test:', u));

---
## 4. Microservices Architecture

### Monolith vs Microservices:
| Aspect | Monolith | Microservices |
|--------|---------|---------------|
| Deployment | Single unit | Independent services |
| Scaling | Scale everything | Scale individual services |
| Technology | Single stack | Polyglot (different languages per service) |
| Complexity | Simple to start | Complex infrastructure |
| Testing | Simple integration | Complex E2E testing |
| Team | Small team, shared codebase | Autonomous teams per service |

### Communication patterns:
- **Synchronous**: HTTP/REST, gRPC (request-response)
- **Asynchronous**: Message queues (RabbitMQ, Kafka), Event bus

### Key patterns:
- **API Gateway** — Single entry point, routes to services
- **Service Discovery** — Services find each other (Consul, etcd)
- **Circuit Breaker** — Prevent cascading failures
- **Saga Pattern** — Distributed transactions across services
- **CQRS** — Separate read and write models

> **Interview Tip:** Don't recommend microservices for everything. Start with a monolith, split into microservices when team/product scale demands it.

In [ ]:
// Circuit Breaker pattern implementation

class CircuitBreaker {
    constructor(fn, options = {}) {
        this.fn = fn;
        this.failureThreshold = options.failureThreshold || 5;
        this.resetTimeout = options.resetTimeout || 30000;
        this.state = 'CLOSED'; // CLOSED, OPEN, HALF_OPEN
        this.failures = 0;
        this.lastFailTime = null;
    }

    async call(...args) {
        if (this.state === 'OPEN') {
            if (Date.now() - this.lastFailTime > this.resetTimeout) {
                this.state = 'HALF_OPEN';
            } else {
                throw new Error('Circuit is OPEN — service unavailable');
            }
        }

        try {
            const result = await this.fn(...args);
            this._onSuccess();
            return result;
        } catch (err) {
            this._onFailure();
            throw err;
        }
    }

    _onSuccess() {
        this.failures = 0;
        this.state = 'CLOSED';
    }

    _onFailure() {
        this.failures++;
        this.lastFailTime = Date.now();
        if (this.failures >= this.failureThreshold) {
            this.state = 'OPEN';
            console.log('Circuit OPENED — too many failures');
        }
    }
}

// Usage
let callCount = 0;
const unreliableService = async () => {
    callCount++;
    if (callCount <= 5) throw new Error('Service down');
    return 'Service response';
};

const breaker = new CircuitBreaker(unreliableService, { failureThreshold: 3, resetTimeout: 1000 });

(async () => {
    for (let i = 0; i < 7; i++) {
        try {
            const result = await breaker.call();
            console.log(`Call ${i + 1}: ${result} [${breaker.state}]`);
        } catch (err) {
            console.log(`Call ${i + 1}: ${err.message} [${breaker.state}]`);
        }
    }
})();

---
## 5. Message Queues

Decouple services and handle async workloads.

### Popular choices:
| Queue | Best for |
|-------|----------|
| **RabbitMQ** | Traditional message broker, routing, reliability |
| **Redis (Bull/BullMQ)** | Job queues, simple pub/sub, caching |
| **Apache Kafka** | Event streaming, high throughput, log-based |
| **AWS SQS** | Managed, serverless-friendly |

### Common use cases:
- Email sending (don't block HTTP response)
- Image/video processing
- Order processing
- Notifications
- Data pipeline processing

### BullMQ example (Redis-based):
```javascript
const { Queue, Worker } = require('bullmq');

// Producer
const emailQueue = new Queue('email');
await emailQueue.add('welcome', {
    to: 'user@example.com',
    subject: 'Welcome!',
    body: 'Thanks for signing up'
});

// Consumer (can run on different server)
const worker = new Worker('email', async (job) => {
    await sendEmail(job.data);
    console.log(`Email sent to ${job.data.to}`);
});
```

---
## 6. WebSockets & Real-Time

### HTTP vs WebSocket:
| Feature | HTTP | WebSocket |
|---------|------|----------|
| Connection | Request/Response (new each time) | Persistent, bidirectional |
| Direction | Client → Server (polling for reverse) | Both directions anytime |
| Overhead | Headers on every request | Small frames after handshake |
| Best for | CRUD APIs, static content | Chat, live updates, gaming |

### Socket.io example:
```javascript
// Server
const { Server } = require('socket.io');
const io = new Server(httpServer, { cors: { origin: '*' } });

io.on('connection', (socket) => {
    console.log('User connected:', socket.id);

    socket.on('chat message', (msg) => {
        io.emit('chat message', msg); // Broadcast to all
    });

    socket.on('join room', (room) => {
        socket.join(room);
        io.to(room).emit('user joined', socket.id);
    });

    socket.on('disconnect', () => {
        console.log('User disconnected:', socket.id);
    });
});

// Client
const socket = io('http://localhost:3000');
socket.emit('chat message', 'Hello everyone!');
socket.on('chat message', (msg) => displayMessage(msg));
```

---
## 7. GraphQL vs REST

| Feature | REST | GraphQL |
|---------|------|--------|
| Endpoints | Multiple (`/users`, `/posts`) | Single (`/graphql`) |
| Data fetching | Fixed response shape | Client specifies exact fields |
| Over-fetching | Common (get entire user object) | Never (request only needed fields) |
| Under-fetching | Common (need multiple requests) | Never (nested queries) |
| Versioning | URL-based (`/v1/`, `/v2/`) | Schema evolution, deprecation |
| Caching | HTTP caching (easy) | More complex (needs client libs) |
| Learning curve | Low | Higher |
| Best for | Simple CRUD, public APIs | Complex frontends, mobile apps |

### GraphQL query example:
```graphql
# Client requests exactly what they need
query {
    user(id: "123") {
        name
        email
        posts(last: 5) {
            title
            commentCount
        }
    }
}
```

> **Interview Tip:** Don't say one is strictly better. REST is simpler and has great HTTP caching. GraphQL solves over/under-fetching for complex UIs. Many companies use both.

---
## 8. TypeScript with Node.js

TypeScript is increasingly expected in Node.js interviews.

### Quick setup:
```bash
npm init -y
npm install -D typescript @types/node ts-node
npx tsc --init
```

### Key TypeScript features for Node.js:
```typescript
// Interfaces for request/response types
interface User {
    id: number;
    name: string;
    email: string;
    role: 'admin' | 'user';  // Union types
    age?: number;             // Optional
}

// Generics for reusable patterns
interface ApiResponse<T> {
    data: T;
    meta: { page: number; total: number };
}

// Type-safe Express handlers
import { Request, Response, NextFunction } from 'express';

interface CreateUserBody {
    name: string;
    email: string;
}

app.post('/users', async (req: Request<{}, {}, CreateUserBody>, res: Response) => {
    const { name, email } = req.body; // Type-safe!
    // ...
});
```

### Benefits in interviews:
- Catches bugs at compile time
- Self-documenting code (types as documentation)
- Better IDE autocomplete
- Required by NestJS, Angular

---
## 9. Node.js Best Practices Summary

### Code Organization:
```
src/
├── config/           # Configuration files
├── controllers/      # Route handlers
├── middleware/        # Custom middleware
├── models/           # Database models
├── routes/           # Route definitions
├── services/         # Business logic
├── utils/            # Helper functions
├── validators/       # Input validation schemas
├── errors/           # Custom error classes
├── tests/            # Test files
├── app.js            # Express app setup
└── server.js         # Server startup
```

### Top 15 Best Practices:
1. **Use async/await** — avoid callback hell
2. **Handle all errors** — every Promise needs a catch
3. **Validate all input** — never trust user data
4. **Use environment variables** — no secrets in code
5. **Use a linter** — ESLint with Airbnb/Standard config
6. **Write tests** — aim for 80%+ coverage on business logic
7. **Use TypeScript** — or at least JSDoc for type safety
8. **Log properly** — structured logging with Pino/Winston
9. **Use `helmet`** — security headers by default
10. **Rate limit APIs** — prevent abuse
11. **Use connection pooling** — for database connections
12. **Cache expensive operations** — Redis for shared cache
13. **Handle graceful shutdown** — close connections on SIGTERM
14. **Use `npm ci` in CI/CD** — reproducible builds
15. **Keep dependencies updated** — `npm audit` regularly

---
## 10. Interview Questions & Answers

### Q1: When would you use Worker Threads vs Child Processes?
**A:** Worker Threads for CPU-intensive JavaScript tasks (shared memory, lower overhead, same V8 instance). Child Processes for running external programs, when you need process isolation, or non-JS tasks. Use `exec`/`spawn` for shell commands, `fork` for Node.js scripts with IPC.

### Q2: Explain the Singleton pattern in Node.js.
**A:** Node.js module caching naturally creates singletons — `require()` caches the exported object after first load. Any subsequent require returns the same instance. So `module.exports = new DatabaseConnection()` gives you a singleton across the entire application.

### Q3: What is a Circuit Breaker pattern?
**A:** It prevents cascading failures in distributed systems. After a threshold of failures, the circuit "opens" and immediately returns errors without calling the failing service (fast-fail). After a timeout, it enters "half-open" state, allowing a test request. If it succeeds, the circuit closes (normal operation resumes).

### Q4: Monolith vs Microservices — how do you decide?
**A:** Start with a monolith — it's simpler to develop, test, and deploy. Consider microservices when: team grows beyond ~10 developers, parts of the app need independent scaling, you want technology diversity, or deployment independence. The cost is operational complexity (networking, distributed tracing, data consistency).

### Q5: How would you implement real-time features in Node.js?
**A:** Use WebSockets (Socket.io or `ws` library) for bidirectional real-time communication. For simpler cases, Server-Sent Events (SSE) for one-way server-to-client updates. For scalability, use Redis pub/sub as a message broker between Socket.io instances across multiple servers.

### Q6: REST vs GraphQL — when to use each?
**A:** REST for simple CRUD APIs, public APIs, and when HTTP caching is important. GraphQL when the frontend needs flexibility in data fetching, has complex nested data requirements, or multiple clients need different data shapes from the same endpoint. Many companies use REST for simple services and GraphQL as an API gateway.

### Q7: What are the benefits of using TypeScript with Node.js?
**A:** Compile-time type checking catches bugs early, interfaces serve as documentation, better IDE support (autocomplete, refactoring), makes large codebases maintainable, required by modern frameworks (NestJS). The trade-off is added build step and learning curve.